# 07 - Export Data

## Export Capstone Data for Local Use

This notebook exports the final cleaned Silver tables and Power BI dashboard datasets to CSV files and creates one ZIP file for local download.

### Silver tables
`stores_silver`, `products_silver`, `suppliers_silver`, `sales_silver`, `inventory_silver`, `purchase_orders_silver`

### Dashboard datasets
`dashboard_sales_monthly`, `dashboard_store_performance`, `dashboard_product_sales`, `dashboard_inventory`, `dashboard_inventory_risk`, `dashboard_supplier_performance`, `dashboard_reorder_point`


In [0]:
pwd


In [0]:
from pyspark.sql import functions as F
import os
import shutil
import glob
import zipfile

BASE_PATH = "/Workspace/Users/yogeshberwal266@gmail.com/Capstone Project"
SILVER_PATH = f"{BASE_PATH}/silver"
DASHBOARD_PATH = f"{BASE_PATH}/dashboard"
ZIP_PATH = "/Workspace/Users/yogeshberwal266@gmail.com/Capstone Project/MegaMart_Capstone_Data.zip"

os.makedirs(SILVER_PATH, exist_ok=True)
os.makedirs(DASHBOARD_PATH, exist_ok=True)

print("Export directory:", BASE_PATH)
print("Silver directory:", SILVER_PATH)
print("Dashboard directory:", DASHBOARD_PATH)
print("Zip file:", ZIP_PATH)


## 1. Export Silver Tables

In [0]:
silver_tables = [
    "stores_silver",
    "products_silver",
    "suppliers_silver",
    "sales_silver",
    "inventory_silver",
    "purchase_orders_silver"
]

for table_name in silver_tables:
    output_dir = f"{SILVER_PATH}/{table_name}"
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, "part-00000.csv")
    (
        spark.table(table_name)
        .toPandas()
        .to_csv(output_file, index=False)
    )
    print(f"Exported: {table_name}")


## 2. Export Dashboard Datasets

In [0]:
# Recreate dashboard temp views from notebook 06_dashboard_data (they are session-scoped)
dashboard_view_sql = [
    """CREATE OR REPLACE TEMP VIEW dashboard_sales_monthly AS
SELECT
    s.store_id,
    s.store_name,
    DATE_FORMAT(sa.sale_date, 'yyyy-MM') AS sales_month,
    SUM(sa.quantity_sold) AS quantity_sold,
    COUNT(DISTINCT sa.transaction_id) AS transaction_count,
    ROUND(SUM(sa.total_amount), 2) AS total_revenue,
    ROUND(SUM(sa.total_amount) / COUNT(DISTINCT sa.transaction_id), 2) AS avg_transaction_value
FROM sales_silver sa
INNER JOIN stores_silver s ON sa.store_id = s.store_id
WHERE YEAR(sa.sale_date) = 2024
GROUP BY s.store_id, s.store_name, DATE_FORMAT(sa.sale_date, 'yyyy-MM')""",
    """CREATE OR REPLACE TEMP VIEW dashboard_store_performance AS
WITH store_metrics AS (
    SELECT
        s.store_id,
        s.store_name,
        SUM(sa.quantity_sold) AS quantity_sold,
        COUNT(DISTINCT sa.transaction_id) AS transaction_count,
        ROUND(SUM(sa.total_amount), 2) AS total_revenue
    FROM sales_silver sa
    INNER JOIN stores_silver s ON sa.store_id = s.store_id
    WHERE YEAR(sa.sale_date) = 2024
    GROUP BY s.store_id, s.store_name
)
SELECT
    store_id,
    store_name,
    quantity_sold,
    transaction_count,
    total_revenue,
    ROUND(total_revenue / transaction_count, 2) AS avg_transaction_value,
    DENSE_RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank,
    DENSE_RANK() OVER (ORDER BY transaction_count DESC) AS transaction_rank
FROM store_metrics""",
    """CREATE OR REPLACE TEMP VIEW dashboard_product_sales AS
SELECT
    p.product_id,
    p.product_name,
    p.category,
    SUM(sa.quantity_sold) AS quantity_sold,
    ROUND(SUM(sa.total_amount), 2) AS total_revenue,
    COUNT(DISTINCT sa.transaction_id) AS transaction_count
FROM sales_silver sa
INNER JOIN products_silver p ON sa.product_id = p.product_id
WHERE YEAR(sa.sale_date) = 2024
GROUP BY p.product_id, p.product_name, p.category""",
    """CREATE OR REPLACE TEMP VIEW dashboard_inventory AS
SELECT
    i.store_id,
    s.store_name,
    i.product_id,
    p.product_name,
    p.category,
    i.quantity_on_hand,
    CASE
        WHEN i.quantity_on_hand = 0 THEN 'Out of Stock'
        WHEN i.quantity_on_hand < 50 THEN 'Low Stock'
        ELSE 'Normal Stock'
    END AS stock_status
FROM inventory_silver i
INNER JOIN stores_silver s ON i.store_id = s.store_id
INNER JOIN products_silver p ON i.product_id = p.product_id""",
    """CREATE OR REPLACE TEMP VIEW dashboard_inventory_risk AS
WITH product_velocity AS (
    SELECT product_id, SUM(quantity_sold) / 365.0 AS avg_daily_sales
    FROM sales_silver
    WHERE YEAR(sale_date) = 2024
    GROUP BY product_id
),
velocity_threshold AS (
    SELECT percentile_approx(avg_daily_sales, 0.5) AS median_daily_sales FROM product_velocity
),
inventory_threshold AS (
    SELECT percentile_approx(quantity_on_hand, 0.5) AS median_inventory FROM inventory_silver
)
SELECT
    i.store_id,
    s.store_name,
    i.product_id,
    p.product_name,
    p.category,
    i.quantity_on_hand,
    ROUND(COALESCE(v.avg_daily_sales, 0), 2) AS avg_daily_sales,
    CASE
        WHEN i.quantity_on_hand = 0 THEN 'Out of Stock'
        WHEN i.quantity_on_hand < 50 THEN 'Low Stock'
        ELSE 'Normal Stock'
    END AS stock_status,
    CASE
        WHEN COALESCE(v.avg_daily_sales, 0) > vt.median_daily_sales AND i.quantity_on_hand < 50
            THEN 'High Velocity - Low Stock'
        WHEN COALESCE(v.avg_daily_sales, 0) < vt.median_daily_sales AND i.quantity_on_hand > it.median_inventory
            THEN 'Slow-Moving - Potential Overstock'
        WHEN COALESCE(v.avg_daily_sales, 0) > vt.median_daily_sales
            THEN 'High Velocity'
        WHEN i.quantity_on_hand > it.median_inventory
            THEN 'Potential Overstock'
        ELSE 'Normal'
    END AS inventory_flag
FROM inventory_silver i
INNER JOIN stores_silver s ON i.store_id = s.store_id
INNER JOIN products_silver p ON i.product_id = p.product_id
LEFT JOIN product_velocity v ON i.product_id = v.product_id
CROSS JOIN velocity_threshold vt
CROSS JOIN inventory_threshold it""",
    """CREATE OR REPLACE TEMP VIEW dashboard_supplier_performance AS
SELECT
    s.supplier_id,
    s.supplier_name,
    COUNT(po.po_id) AS total_orders,
    SUM(CASE WHEN po.delivery_date IS NOT NULL THEN 1 ELSE 0 END) AS completed_orders,
    ROUND(100.0 * SUM(CASE WHEN po.delivery_date IS NOT NULL THEN 1 ELSE 0 END) / COUNT(po.po_id), 2) AS completion_rate_pct,
    ROUND(AVG(CASE WHEN po.delivery_date IS NOT NULL THEN DATEDIFF(po.delivery_date, po.order_date) END), 2) AS avg_delivery_days
FROM purchase_orders_silver po
INNER JOIN suppliers_silver s ON po.supplier_id = s.supplier_id
GROUP BY s.supplier_id, s.supplier_name""",
    """CREATE OR REPLACE TEMP VIEW dashboard_reorder_point AS
WITH sales_velocity AS (
    SELECT store_id, product_id, SUM(quantity_sold) / 365.0 AS avg_daily_sales
    FROM sales_silver
    WHERE YEAR(sale_date) = 2024
    GROUP BY store_id, product_id
),
supplier_lead_time AS (
    SELECT supplier_id, AVG(DATEDIFF(delivery_date, order_date)) AS avg_lead_time_days
    FROM purchase_orders_silver
    WHERE delivery_date IS NOT NULL
    GROUP BY supplier_id
)
SELECT
    i.store_id,
    s.store_name,
    i.product_id,
    p.product_name,
    p.category,
    i.quantity_on_hand,
    p.supplier_id,
    ROUND(COALESCE(v.avg_daily_sales, 0), 2) AS avg_daily_sales,
    ROUND(COALESCE(sl.avg_lead_time_days, 0), 2) AS avg_supplier_lead_time_days,
    ROUND(COALESCE(v.avg_daily_sales, 0) * COALESCE(sl.avg_lead_time_days, 0), 2) AS baseline_reorder_point,
    CASE
        WHEN i.quantity_on_hand <= COALESCE(v.avg_daily_sales, 0) * COALESCE(sl.avg_lead_time_days, 0)
            THEN 'Reorder Recommended'
        ELSE 'Stock Above Reorder Point'
    END AS reorder_status
FROM inventory_silver i
INNER JOIN stores_silver s ON i.store_id = s.store_id
INNER JOIN products_silver p ON i.product_id = p.product_id
LEFT JOIN sales_velocity v ON i.store_id = v.store_id AND i.product_id = v.product_id
LEFT JOIN supplier_lead_time sl ON p.supplier_id = sl.supplier_id""",
]

for view_sql in dashboard_view_sql:
    spark.sql(view_sql)

dashboard_tables = [
    "dashboard_sales_monthly",
    "dashboard_store_performance",
    "dashboard_product_sales",
    "dashboard_inventory",
    "dashboard_inventory_risk",
    "dashboard_supplier_performance",
    "dashboard_reorder_point"
]

for table_name in dashboard_tables:
    output_dir = f"{DASHBOARD_PATH}/{table_name}"
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, "part-00000.csv")
    (
        spark.table(table_name)
        .toPandas()
        .to_csv(output_file, index=False)
    )
    print(f"Exported: {table_name}")


## 3. Convert Spark CSV Folders into Normal CSV Files

In [0]:
def flatten_csv_exports(source_dir, table_names, target_dir):
    for table_name in table_names:
        table_dir = os.path.join(source_dir, table_name)
        csv_files = glob.glob(os.path.join(table_dir, "part-*.csv"))

        if not csv_files:
            raise FileNotFoundError(f"No CSV file found for {table_name}")

        target_file = os.path.join(target_dir, f"{table_name}.csv")
        shutil.copy2(csv_files[0], target_file)
        shutil.rmtree(table_dir)

        print(f"Created: {target_file}")

flatten_csv_exports(SILVER_PATH, silver_tables, SILVER_PATH)
flatten_csv_exports(DASHBOARD_PATH, dashboard_tables, DASHBOARD_PATH)


## 4. Verify Exported Files

In [0]:
print("Silver exports:")
for table_name in silver_tables:
    path = f"{SILVER_PATH}/{table_name}.csv"
    print(f"  {table_name}.csv | {os.path.getsize(path):,} bytes")

print("\nDashboard exports:")
for table_name in dashboard_tables:
    path = f"{DASHBOARD_PATH}/{table_name}.csv"
    print(f"  {table_name}.csv | {os.path.getsize(path):,} bytes")


## 5. Create ZIP File

In [0]:
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for folder_name, folder_path in [
        ("silver", SILVER_PATH),
        ("dashboard", DASHBOARD_PATH)
    ]:
        for filename in os.listdir(folder_path):
            file_path = os.path.join(folder_path, filename)
            if os.path.isfile(file_path):
                zipf.write(file_path, f"{folder_name}/{filename}")

print("ZIP created:", ZIP_PATH)


## 6. Download the ZIP

In [0]:
displayHTML(
    '<a href="/files/megamart_capstone_export/MegaMart_Capstone_Data.zip" '
    'target="_blank">Download MegaMart Capstone Data ZIP</a>'
)


## Final ZIP Structure

```text
MegaMart_Capstone_Data.zip
├── silver/
│   ├── stores_silver.csv
│   ├── products_silver.csv
│   ├── suppliers_silver.csv
│   ├── sales_silver.csv
│   ├── inventory_silver.csv
│   └── purchase_orders_silver.csv
│
└── dashboard/
    ├── dashboard_sales_monthly.csv
    ├── dashboard_store_performance.csv
    ├── dashboard_product_sales.csv
    ├── dashboard_inventory.csv
    ├── dashboard_inventory_risk.csv
    ├── dashboard_supplier_performance.csv
    └── dashboard_reorder_point.csv
```

The Bronze layer is intentionally not exported because the Silver layer is the cleaned analytical layer used by the project and Power BI.
